## Probing: can frozen encodings identify PathMNIST tissue types?

PathMNIST patches are already pre-cut and single-labeled (one of 9 colon-tissue classes), which makes it the simplest sanity check for what a frozen pathology encoder actually captures: fit a small classifier on top of the (frozen) UNI2-h / Virchow2 embeddings and see how well it recovers the tissue label -- no fine-tuning of the encoder itself.

**The probe:** a *linear MLP* -- an MLP with zero hidden layers, i.e. a single `nn.Linear` layer mapping embedding -> 9 class logits, trained with backprop (softmax + cross-entropy + gradient descent). This is the same model `sklearn.linear_model.LogisticRegression` fits under the hood, so it answers the same "linearly separable?" question -- the difference is you write the training loop yourself instead of calling `.fit()`, which is the point of this notebook.

Before probing the real 1536-d / 2560-d embeddings, the next section builds and trains a linear MLP on a tiny synthetic 2D dataset -- small enough to actually plot the decision boundary -- so the model/loss/training-loop pattern is on screen once before you write it yourself for PathMNIST.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from pathlib import Path

import torch
import torch.nn as nn
from nbhelper import (
    np,
    pd,
    plt,
)
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from probing import plot_confusion_matrix, probe_metrics  # noqa: E402

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

### 1. Load encodings

Loads the shared, full-release encodings from `/home/shared/data/pathmnist/encodings/` -- produced by `00_preparation/01_encodings/encode_pathmnist.ipynb` with `USE_SUBSET = False` -- rather than each participant's own (possibly subset) run, so probing here always sees the complete PathMNIST release. Each `.npz` holds per-split features and labels together.

In [ ]:
pathmnist_base_dir = Path("/home/shared/data/pathmnist/")
encodings_dir = Path("/home/shared/data/pathmnist/encodings")

LABEL_NAMES = {
    0: "adipose",
    1: "background",
    2: "debris",
    3: "lymphocytes",
    4: "mucus",
    5: "smooth muscle",
    6: "normal colon mucosa",
    7: "cancer-associated stroma",
    8: "colorectal adenocarcinoma epithelium",
}
CLASS_NAMES = [LABEL_NAMES[i] for i in range(len(LABEL_NAMES))]

MODEL_NAMES = ["uni2-h", "virchow2"]
SPLITS = ["train", "val", "test"]

encodings = {}  # encodings[model][split] -> (X, y)
for model_name in MODEL_NAMES:
    npz_path = encodings_dir / f"{model_name}_full.npz"
    data = np.load(npz_path)
    encodings[model_name] = {
        split: (data[f"{split}_encodings"], data[f"{split}_labels"]) for split in SPLITS
    }
    print(
        f"{model_name:10s} <- {npz_path.name}  "
        f"(train={len(encodings[model_name]['train'][1]):,}, "
        f"test={len(encodings[model_name]['test'][1]):,})"
    )

### 2. A linear MLP, from scratch, on a toy example

A linear MLP for classification is: one `nn.Linear(in_dim, n_classes)` layer producing raw logits, `nn.CrossEntropyLoss` (softmax + negative log-likelihood in one step), and a plain gradient-descent training loop. No hidden layers -- that's what makes it *linear*: the decision boundary between any two classes is a straight line (a hyperplane, in higher dimensions), exactly like logistic regression.

Built and trained here on a synthetic 4-blob 2D dataset (`sklearn.datasets.make_blobs`) so the model, loss, and training loop are visible end to end, and the resulting decision boundary can actually be plotted -- something impossible in the real embeddings' 1536-d / 2560-d space.

In [ ]:
from sklearn.datasets import make_blobs


class LinearMLP(nn.Module):
    """An MLP with zero hidden layers -- one nn.Linear straight from input to
    class logits. Multinomial logistic regression in everything but name; the
    difference is this one is trained with an explicit backprop loop."""

    def __init__(self, in_dim: int, n_classes: int):
        super().__init__()
        self.linear = nn.Linear(in_dim, n_classes)

    def forward(self, x):
        return self.linear(x)  # raw logits -- CrossEntropyLoss applies softmax itself


def train_linear_mlp(X_train, y_train, n_classes, epochs=300, lr=0.05, seed=42, device="cpu"):
    torch.manual_seed(seed)
    model = LinearMLP(X_train.shape[1], n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    X = torch.as_tensor(X_train, dtype=torch.float32, device=device)
    y = torch.as_tensor(y_train, dtype=torch.long, device=device)

    losses = []
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model, losses


@torch.no_grad()
def predict_linear_mlp(model, X, device="cpu"):
    model.eval()
    X = torch.as_tensor(X, dtype=torch.float32, device=device)
    return model(X).argmax(dim=1).cpu().numpy()


# --- toy example: 4 classes, 2D, so the decision boundary can be plotted ---
X_toy, y_toy = make_blobs(n_samples=400, centers=4, cluster_std=1.8, random_state=42)
toy_scaler = StandardScaler().fit(X_toy)
X_toy_scaled = toy_scaler.transform(X_toy)

toy_model, toy_losses = train_linear_mlp(X_toy_scaled, y_toy, n_classes=4, device=device)
toy_acc = (predict_linear_mlp(toy_model, X_toy_scaled, device=device) == y_toy).mean()
print(f"toy linear MLP -- final loss={toy_losses[-1]:.3f}  train accuracy={toy_acc:.3f}")

# --- visualize: loss curve + decision boundary ---
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))

axs[0].plot(toy_losses)
axs[0].set_xlabel("epoch")
axs[0].set_ylabel("cross-entropy loss")
axs[0].set_title("training loss")

xx, yy = np.meshgrid(
    np.linspace(X_toy_scaled[:, 0].min() - 1, X_toy_scaled[:, 0].max() + 1, 200),
    np.linspace(X_toy_scaled[:, 1].min() - 1, X_toy_scaled[:, 1].max() + 1, 200),
)
grid_preds = predict_linear_mlp(toy_model, np.c_[xx.ravel(), yy.ravel()], device=device)
axs[1].contourf(xx, yy, grid_preds.reshape(xx.shape), alpha=0.3, cmap="tab10")
axs[1].scatter(
    X_toy_scaled[:, 0],
    X_toy_scaled[:, 1],
    c=y_toy,
    cmap="tab10",
    s=12,
    edgecolor="k",
    linewidth=0.3,
)
axs[1].set_title("decision boundary (straight lines -- it's linear)")

plt.tight_layout()
plt.show()

### 3. Exercise: fit the linear MLP probe on the real encodings

Now do for the real PathMNIST embeddings what the toy example just did for 2D blobs. Reuse `LinearMLP`, `train_linear_mlp`, and `predict_linear_mlp` from above -- nothing new to invent, just apply them at the real scale (1536-d / 2560-d, 9 classes, tens of thousands of patches instead of 400).

For each `model_name` in `MODEL_NAMES`:
1. Get `X_train, y_train = encodings[model_name]["train"]` and `X_test, y_test = encodings[model_name]["test"]`.
2. Standardize with a `StandardScaler` **fit on `X_train` only** (never fit preprocessing on test data), and transform both splits.
3. Train with `train_linear_mlp(X_train_scaled, y_train, n_classes=len(LABEL_NAMES), device=device)`.
4. Predict on `X_test_scaled` with `predict_linear_mlp`, and score with `probe_metrics(y_test, y_pred, labels=list(LABEL_NAMES))` (from `probing.py` -- gives you accuracy, macro-F1, and the confusion matrix in one call).

Build the two variables the rest of this notebook expects:
- `results_df`: a `pd.DataFrame` with one row per encoder (`accuracy`, `macro_f1` columns), indexed by `encoder`.
- `confusions`: a dict `{model_name: confusion_matrix}`.

A few hundred epochs (as in the toy example) is plenty for a linear model even at this scale -- this should take seconds per encoder on GPU, well under a minute on CPU.

In [ ]:
results = []
confusions = {}

for model_name in MODEL_NAMES:
    X_train, y_train = encodings[model_name]["train"]
    X_test, y_test = encodings[model_name]["test"]

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model, losses = train_linear_mlp(
        X_train_scaled, y_train, n_classes=len(LABEL_NAMES), device=device
    )
    y_pred = predict_linear_mlp(model, X_test_scaled, device=device)
    metrics = probe_metrics(y_test, y_pred, labels=list(LABEL_NAMES))

    results.append(
        {"encoder": model_name, "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]}
    )
    confusions[(model_name, "linear")] = metrics["confusion_matrix"]

results_df = pd.DataFrame(results).set_index("encoder")
results_df

### 4. Confusion matrices (linear MLP probe)

Where do the mistakes fall? Morphologically similar classes (e.g. cancer-associated stroma vs. smooth muscle, or debris vs. background) are the classic confusions to look for.

In [ ]:
fig, axs = plt.subplots(1, len(MODEL_NAMES), figsize=(8 * len(MODEL_NAMES), 7))
for ax, model_name in zip(axs, MODEL_NAMES):
    plot_confusion_matrix(
        confusions[(model_name, "linear")],
        CLASS_NAMES,
        ax,
        title=f"{model_name} -- linear MLP probe",
    )
plt.tight_layout()
plt.show()

#### What about mislabeled images? How do they look?

### 5. Embedding geometry

PCA to 2D, colored by tissue class, plus a silhouette score (quantifies how tightly each class clusters vs. how well-separated it is from the others). Both are computed on a random 5,000-patch subsample of the train split, not the full ~90k -- `silhouette_score` is O(n^2) in the number of samples, so the full split would be far slower for no real gain in what the plot shows.

In [ ]:
N_GEOMETRY_SAMPLES = 5_000  # silhouette_score is O(n^2) -- subsample for speed

fig, axs = plt.subplots(1, len(MODEL_NAMES), figsize=(7 * len(MODEL_NAMES), 6))
for ax, model_name in zip(axs, MODEL_NAMES):
    X_train, y_train = encodings[model_name]["train"]
    sample_idx = np.random.RandomState(42).choice(
        len(X_train), size=min(N_GEOMETRY_SAMPLES, len(X_train)), replace=False
    )
    X_sample, y_sample = X_train[sample_idx], y_train[sample_idx]

    coords = PCA(n_components=2, random_state=42).fit_transform(X_sample)
    sil = silhouette_score(X_sample, y_sample)

    for label, name in LABEL_NAMES.items():
        mask = y_sample == label
        ax.scatter(coords[mask, 0], coords[mask, 1], s=6, alpha=0.5, label=name)
    ax.set_title(f"{model_name} -- PCA(2), silhouette={sil:.3f}  (n={len(X_sample):,})")

axs[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

### Takeaways

- If linear-MLP accuracy is well above the ~11% random-guess baseline for 9 classes, tissue type is close to *linearly* readable straight out of the frozen encoder -- no fine-tuning needed for this task. The toy example above showed what "linearly readable" looks like geometrically: straight decision boundaries between classes.
- The confusion matrix's off-diagonal mass and the PCA plot's cluster overlap should agree: whichever classes are confused are also the ones sitting close together in the 2D projection.
- Compare the two encoders' accuracy and where their mistakes land -- do they confuse the same tissue pairs, or different ones?
- This is the easiest of the three datasets (single institution, patches pre-cut, no domain shift) -- treat it as a baseline before the harder probes in the next notebooks.